In [1]:
import sys
import os
from os.path import join
import glob

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display, clear_output

sys.path.insert(0, "/home/2649/repos/ABC-SN/code")
import abcsn_config
import abcsn_training
import data_degrading as dg
import data_plotting as dplt
import data_preparation as dp
import preprocessing

from icecream import ic
from importlib import reload

REPO_DIR = "/home/2649/repos/spec_res"
DATA_DIR = join(REPO_DIR, "data")
BIANCO_DENOISING_DIR = join(DATA_DIR, "original_resolution_parquet/bianco_denoising")

2026-02-09 16:44:26.275991: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-09 16:44:26.294032: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-09 16:44:26.299783: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-09 16:44:26.314299: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-09 16:44:28.269411: W tensorflow/compiler/tf2

In [2]:
file_alldata = join(DATA_DIR, "forSNR", "alldata.parquet")
df_alldata = pd.read_parquet(file_alldata)
df_alldata.reset_index(inplace=True)

In [3]:
metadata_cols = [
    "SN Name",
    "SN Subtype",
    "SN Subtype ID",
    "SN Maintype",
    "SN Maintype ID",
    "Spectral Phase",
]
df_metadata = df_alldata[metadata_cols].copy(deep=True)
df_metadata

,SN Name,SN Subtype,SN Subtype ID,SN Maintype,SN Maintype ID,Spectral Phase
0,sn2008ar,Ia-norm,0,Ia,0,-8.5
1,sn2008ar,Ia-norm,0,Ia,0,-7.5
2,sn2008ar,Ia-norm,0,Ia,0,-6.6
3,sn2008ar,Ia-norm,0,Ia,0,-4.6
4,sn2008ar,Ia-norm,0,Ia,0,-3.7
...,...,...,...,...,...,...
3759,sn2003jd,Ic-broad,8,Ic,2,46.4
3760,sn2003jd,Ic-broad,8,Ic,2,47.4
3761,sn2003jd,Ic-broad,8,Ic,2,48.4
3762,sn2003jd,Ic-broad,8,Ic,2,49.4


In [4]:
df_data = df_alldata.drop(columns=metadata_cols).copy(deep=True)
df_data

,2501.69,2505.08,2508.48,2511.87,2515.28,2518.69,2522.1,2525.51,2528.94,2532.36,...,9872.21,9885.59,9898.98,9912.39,9925.82,9939.27,9952.73,9966.21,9979.71,9993.24
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3759,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3760,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3761,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3762,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
df_data.to_parquet(join(DATA_DIR, "forSNR", "data.parquet"))
df_metadata.to_parquet(join(DATA_DIR, "forSNR", "metadata.parquet"))

In [6]:
pd.read_parquet(join(DATA_DIR, "forSNR", "data.parquet"))

,2501.69,2505.08,2508.48,2511.87,2515.28,2518.69,2522.1,2525.51,2528.94,2532.36,...,9872.21,9885.59,9898.98,9912.39,9925.82,9939.27,9952.73,9966.21,9979.71,9993.24
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3759,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3760,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3761,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3762,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
pd.read_parquet(join(DATA_DIR, "forSNR", "metadata.parquet"))

,SN Name,SN Subtype,SN Subtype ID,SN Maintype,SN Maintype ID,Spectral Phase
0,sn2008ar,Ia-norm,0,Ia,0,-8.5
1,sn2008ar,Ia-norm,0,Ia,0,-7.5
2,sn2008ar,Ia-norm,0,Ia,0,-6.6
3,sn2008ar,Ia-norm,0,Ia,0,-4.6
4,sn2008ar,Ia-norm,0,Ia,0,-3.7
...,...,...,...,...,...,...
3759,sn2003jd,Ic-broad,8,Ic,2,46.4
3760,sn2003jd,Ic-broad,8,Ic,2,47.4
3761,sn2003jd,Ic-broad,8,Ic,2,48.4
3762,sn2003jd,Ic-broad,8,Ic,2,49.4
